# 04 - Orquestación con Databricks Jobs
Despliega y sincroniza `Bronze -> Silver`, con `Gold + ML` opcional (`include_ml=true`).
Cada catálogo tiene un Job independiente. Por defecto solo se despliega; `run_job=true`
inicia la ejecución y espera el resultado final, propagando cualquier fallo.
Se admite serverless o un cluster existente con las dependencias preparadas.

## 1. Configuración
Use el Git Folder completo y seleccione un catálogo preparado con el notebook de setup.
Configure dependencias en el entorno serverless o en el cluster existente antes de ejecutar ML.

In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

from datetime import timedelta

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.dropdown("include_ml", "false", ["false", "true"], "Incluir Gold y entrenamiento")
dbutils.widgets.dropdown("run_job", "false", ["false", "true"], "Ejecutar después de desplegar")
dbutils.widgets.dropdown("enforce_reference_counts", "true", ["true", "false"], "Conteos de referencia")
dbutils.widgets.text("max_vehicle_orphans", "3", "Tolerancia de huérfanos")
dbutils.widgets.text("existing_cluster_id", "", "Cluster existente; vacío para serverless")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
INCLUDE_ML = parse_bool(dbutils.widgets.get("include_ml"))
RUN_JOB = parse_bool(dbutils.widgets.get("run_job"))
ENFORCE_COUNTS = str(parse_bool(dbutils.widgets.get("enforce_reference_counts"))).lower()
MAX_ORPHANS = str(nonnegative_int(dbutils.widgets.get("max_vehicle_orphans")))
CLUSTER_ID = dbutils.widgets.get("existing_cluster_id").strip()
JOB_NAME = f"accidentalidad_barranquilla_pipeline_{PROY_CATALOG}"
# Rutas de Workspace API: no incluyen el prefijo local /Workspace.
WORKSPACE_ROOT = str(PROJECT_ROOT)
if WORKSPACE_ROOT.startswith("/Workspace/"):
    WORKSPACE_ROOT = WORKSPACE_ROOT[len("/Workspace"):]
print(f"Job: {JOB_NAME}; raíz: {WORKSPACE_ROOT}")

## 2. Cliente y notebooks
Resuelve únicamente objetos de tipo notebook. Los errores de permisos o conectividad se
propagan; solo una ruta inexistente permite probar el siguiente candidato.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.jobs import JobSettings
from databricks.sdk.service.workspace import ObjectType

w = WorkspaceClient()

def resolver_notebook(ruta_base):
    candidatos = [ruta_base, f"{ruta_base}.ipynb"]
    for candidato in candidatos:
        try:
            estado = w.workspace.get_status(candidato)
        except NotFound:
            continue
        if estado.object_type == ObjectType.NOTEBOOK:
            return candidato
    raise ValueError(f"No se encontró un notebook en {candidatos}")

BRONZE_NOTEBOOK = resolver_notebook(f"{WORKSPACE_ROOT}/01_Exploracion_y_Transformacion/01_Ingesta_Bronze")
SILVER_NOTEBOOK = resolver_notebook(f"{WORKSPACE_ROOT}/01_Exploracion_y_Transformacion/02_Transformacion_Silver")

## 3. Definición del DAG
Las tareas reciben los parámetros del catálogo y calidad. Incluyen timeout y reintentos
acotados; ML no tiene reintentos automáticos para evitar registros de versiones duplicados.

In [ ]:
def notebook_task(key, path, parameters, depends_on=None):
    task = {
        "task_key": key, "timeout_seconds": 3600, "max_retries": 1,
        "min_retry_interval_millis": 60000, "retry_on_timeout": False,
        "notebook_task": {"notebook_path": path, "source": "WORKSPACE", "base_parameters": parameters},
    }
    if depends_on:
        task.update({"depends_on": [{"task_key": depends_on}], "run_if": "ALL_SUCCESS"})
    if CLUSTER_ID:
        task["existing_cluster_id"] = CLUSTER_ID
    return task

common = {"catalog_name": PROY_CATALOG}
tasks = [
    notebook_task("bronze_ingestion", BRONZE_NOTEBOOK, {**common, "enforce_reference_counts": ENFORCE_COUNTS}),
    notebook_task("silver_transformation", SILVER_NOTEBOOK, {**common, "max_vehicle_orphans": MAX_ORPHANS}, "bronze_ingestion"),
]
if INCLUDE_ML:
    ml_path = resolver_notebook(f"{WORKSPACE_ROOT}/05_Machine_Learning/05_Machine_Learning")
    ml_task = notebook_task("gold_and_training", ml_path, {**common, "promote_champion": "false"}, "silver_transformation")
    # El registro crea versiones: evitar duplicarlas mediante reintentos automáticos.
    ml_task["max_retries"] = 0
    tasks.append(ml_task)
job_settings = JobSettings.from_dict({
    "name": JOB_NAME, "max_concurrent_runs": 1, "timeout_seconds": 10800,
    "tags": {"project": "accidentalidad-databricks", "catalog": PROY_CATALOG}, "tasks": tasks,
})

## 4. Creación o actualización del Job
La definición versionada reemplaza la configuración del Job del mismo catálogo. No se
sobrescribe durante una ejecución activa. Dos Jobs con el mismo nombre producen un error.
El Job anterior sin sufijo de catálogo no se migra ni elimina automáticamente.

In [ ]:
from accidentalidad.jobs import deploy_job

# Job independiente por catálogo; el Job antiguo sin sufijo no se modifica.
JOB_ID = deploy_job(w, JOB_NAME, job_settings)
print(f"[OK] Definición sincronizada: {w.config.host}/jobs/{JOB_ID}")

## 5. Verificación de la definición

Se consulta nuevamente el Job desde Databricks para dejar evidencia de sus tareas y dependencias.

In [ ]:
job_info = w.jobs.get(JOB_ID)

print(f"Job: {job_info.settings.name}")
print(f"Job ID: {JOB_ID}")
print("\nDAG:")

for task in job_info.settings.tasks:
    dependencias = (
        [dep.task_key for dep in task.depends_on]
        if task.depends_on
        else []
    )

    print(
        f"- {task.task_key} | "
        f"depends_on={dependencias or 'ninguna'} | "
        f"run_if={task.run_if or 'ALL_SUCCESS'}"
    )

## 6. Ejecución opcional y verificación
`run_job=true` dispara el Job, imprime su enlace y espera hasta su estado final.
Un fallo o un tiempo de espera agotado hace fallar este notebook; iniciar una ejecución
no se considera evidencia de éxito.

In [ ]:
if RUN_JOB:
    from databricks.sdk.service.jobs import RunResultState
    waiter = w.jobs.run_now(job_id=JOB_ID)
    RUN_ID = waiter.response.run_id
    print(f"Run: {w.config.host}/jobs/{JOB_ID}/runs/{RUN_ID}", flush=True)
    last_state = None
    def progress(run):
        global last_state
        state = str(run.state)
        if state != last_state:
            print(f"Estado: {state}", flush=True)
            last_state = state
    completed = waiter.result(timeout=timedelta(hours=3, minutes=5), callback=progress)
    if completed.state.result_state != RunResultState.SUCCESS:
        raise RuntimeError(f"La ejecución no terminó exitosamente: {completed.state}")
    print(f"[OK] Ejecución finalizada: {RUN_ID}")
else:
    print("Definición desplegada; run_job=false, no se inició ninguna ejecución")

## 7. Operación
Revise las tareas y el resultado en Jobs & Pipelines, junto con `quality.check_results`.
Los fallos críticos de calidad impiden ejecutar las etapas dependientes. Entrenamiento
no promueve automáticamente un modelo. Los CSV deben permanecer inmutables y debe haber
un único escritor por catálogo (también para ejecuciones manuales). Configure notificaciones
y una identidad de servicio según las políticas de su workspace.